# Week 4 Problem Set: Does This Message Move Voters?

**Instructions:** Complete all four tasks.

You have the results from a survey experiment that tested four persuasion messages. Your job: analyze the results, stress-test the "winning" message, and write a recommendation.


### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/'
                 'main/weeks/wk04_message_testing/data/survey_experiment.csv')
df.shape

## Task 1: Compute ATEs for all four message arms

Compute the mean of `vote_for_candidate` for each arm. Then compute the difference between each treatment arm and the control arm.


In [ ]:
# Compute the control mean
control_mean = df[df['arm'] == 'control']['vote_for_candidate'].mean()
print(f'Control mean: {control_mean:.3f}')
print()

# Compute the ATE for each treatment arm
for arm in ['economy', 'healthcare', 'character', 'abortion']:
    arm_mean = df[df['arm'] == arm]['vote_for_candidate'].mean()
    diff = arm_mean - control_mean
    print(f'{arm}: {arm_mean:.3f}  (ATE: {diff:+.4f})')

**Question 1:** Which arm has the largest positive ATE? How large is it in percentage points?

*Your answer:*


## Task 2: Randomization inference on the abortion arm

Run the two cells below to set up the subset and compute the observed ATE. Then **write the RI loop yourself** using the scaffolded cell that follows.

In [ ]:
# Subset to control and abortion arms
subset = df[df['arm'].isin(['control', 'abortion'])].copy()

# Observed ATE
observed_ate = (
    subset[subset['arm'] == 'abortion']['vote_for_candidate'].mean()
    - subset[subset['arm'] == 'control']['vote_for_candidate'].mean()
)
print(f'Observed ATE: {observed_ate:.4f}')

Now write the randomization inference loop yourself. The cell below has the structure — fill in the three lines marked `# YOUR CODE HERE`.

**Important:** this cell will not run until you fill in all three blanks. A `SyntaxError` means you still have a `# YOUR CODE HERE` placeholder.

*Check: your two-sided p-value should be around 0.03–0.05.*

In [ ]:
np.random.seed(42)
fake_ates = []

for i in range(1000):
    # Step 1: shuffle the arm labels.
    # Use the same .sample(frac=1).values pattern from W3 and from this week's livecode.
    shuffled_labels = # YOUR CODE HERE

    # Step 2: compute the fake ATE (mean of fake treated minus mean of fake control)
    fake_treated = subset['vote_for_candidate'][shuffled_labels == 'abortion']
    fake_control = subset['vote_for_candidate'][shuffled_labels == 'control']
    fake_ate = # YOUR CODE HERE

    fake_ates.append(fake_ate)

# Step 3: compute the two-sided p-value -- the fraction of fake ATEs whose
# ABSOLUTE value is at least the observed |ATE| (same convention as W3).
p_value = # YOUR CODE HERE
print(f'p-value: {p_value:.3f}')

**Question 2:** What is the p-value? In one sentence, interpret what it means.

*Your answer:*


## Task 3: The multiple-comparisons simulation

Suppose all four messages have zero real effect. How often would at least one arm look "significant" by chance?

We will build this in two steps, just like the RI loop.


**Step 1: Derive the significance threshold.**

We already have the null distribution from Task 2. The 95th percentile of the absolute fake ATEs tells us how large a difference needs to be before we would call it "significant."


In [ ]:
# Derive the threshold from the RI null distribution
threshold = np.percentile([abs(f) for f in fake_ates], 95)
print(f'Significance threshold: {threshold:.4f}')
print(f'Any ATE larger than this (in absolute value) is in the most extreme 5% under the null.')

**Step 2: One fake experiment.**

Generate 5,000 outcomes with no treatment effect. Split into 5 groups (control + 4 treatment arms). Check if any treatment arm crosses the threshold.


In [ ]:
# One fake experiment with no treatment effect
np.random.seed(42)
fake_votes = np.random.binomial(1, 0.44, size=5000)

# First 1000 = control
fake_control_mean = fake_votes[:1000].mean()

# Check each of the 4 treatment arms
any_significant = False
for j in range(4):
    arm_name = ['economy', 'healthcare', 'character', 'abortion'][j]
    arm_start = 1000 * (j + 1)
    arm_end = arm_start + 1000
    arm_mean = fake_votes[arm_start:arm_end].mean()
    diff = abs(arm_mean - fake_control_mean)
    sig = '***' if diff >= threshold else ''
    print(f'{arm_name}: diff = {diff:.4f} {sig}')
    if diff >= threshold:
        any_significant = True

print(f'\nAt least one significant? {any_significant}')

**Step 3: Now do that 1,000 times.**

Wrap the fake experiment in a loop. Track how often at least one arm crosses the threshold.


In [ ]:
# 1,000 fake experiments, each with no treatment effect
# For each one, check if any of the 4 arms looks "significant"
np.random.seed(42)
false_positive_count = 0

for i in range(1000):
    fake_votes = np.random.binomial(1, 0.44, size=5000)
    fake_control_mean = fake_votes[:1000].mean()
    
    any_sig = False
    for j in range(4):
        arm_mean = fake_votes[1000*(j+1) : 1000*(j+2)].mean()
        if abs(arm_mean - fake_control_mean) >= threshold:
            any_sig = True
            break
    
    if any_sig:
        false_positive_count += 1

print(f'Fake experiments with at least one "significant" arm: {false_positive_count} / 1000')
print(f'False positive rate: {false_positive_count / 1000:.3f}')

**Question 3:** What fraction of fake experiments produced at least one "significant" arm? In one sentence, explain what this means for the consultant's claim that Message D "won."

*Your answer:*


**Before you move on:** The simulation above tested 4 message arms. If the consultant had tested 10 messages instead of 4 (all with zero true effect), would you expect the false positive rate to go up, go down, or stay the same? Why?

Write your answer in 1–2 sentences in the cell below.

**Your answer:**

*Replace this text with your answer.*

## Task 4: Memo to the campaign manager (250--350 words)

The consultant recommends scaling Message D based on the survey experiment. The campaign manager wants your assessment.

Write a memo addressed to the campaign manager. Your memo must address all three of the following:

**(a) The winner's curse.** The survey experiment tested four messages and Message D had the largest ATE. Is the +4.9-point estimate trustworthy? Use a specific number from your simulation (Task 3) to support your argument.

**(b) The measurement gap.** The survey experiment measured vote *intention*, not actual vote *choice*. Does a survey-measured effect predict what will happen when the message is delivered as a real TV ad? Name one specific reason it might not.

**(c) Steelman.** Construct the strongest version of the argument for scaling Message D anyway. What would a reasonable person say in favor of going with the survey result? Then say whether you accept that argument or not, and why.

**Style rules:**
- Write as a professional memo, not an essay. Use "I recommend" or "I advise against," not "one might consider."
- Use at least one specific number from your analysis.

*Your memo:*


---

## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished — fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't — every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.